## 1. — Configure the source and schema

In [0]:

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
)

SOURCE_PATH = (
    "/Volumes/workspace/bronze/"
    "logistic_landing/tracking_events/"
)

CHECKPOINT_PATH = (
    "/Volumes/workspace/bronze/"
    "pipeline_state/tracking_events_checkpoint"
)

TARGET_TABLE = "workspace.bronze.tracking_events"

event_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("shipment_id", StringType(), True),
    StructField("event_timestamp", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("distribution_center", StringType(), True),
    StructField("source_system", StringType(), True),
    StructField("_rescued_data", StringType(), True),
])

print("Source:", SOURCE_PATH)
print("Target:", TARGET_TABLE)


## 2 — Read JSON files using Auto Loader

In [0]:

events_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .schema(event_schema)
    .load(SOURCE_PATH)
)

bronze_df = (
    events_df
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("source_file", F.col("_metadata.file_path"))
)

print("Streaming source configured successfully")


## 3 — Write the stream into Bronze

In [0]:

query = (
    bronze_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(availableNow=True)
    .toTable(TARGET_TABLE)
)

query.awaitTermination()

print("Auto Loader run completed")


## 4 — Validate the first ingestion

In [0]:
result = spark.sql("""
    SELECT
        COUNT(*) AS total_events,
        COUNT(DISTINCT event_id) AS unique_events
    FROM workspace.bronze.tracking_events
""")

display(result)

## 5 — Check the second JSON batch

In [0]:
CHECKPOINT_PATH = (
    "/Volumes/workspace/bronze/"
    "pipeline_state/tracking_events_checkpoint"
)

TARGET_TABLE = "workspace.bronze.tracking_events"

In [0]:
%sql
SELECT
    COUNT(*) AS total_events,
    COUNT(DISTINCT event_id) AS unique_events
FROM workspace.bronze.tracking_events;